In [ ]:
%load_ext autoreload
%autoreload 2

import json
import pandas as pd
from matplotlib import pyplot as plt
from utils.display import display_grid
import numpy as np

import os
if "original_dir" not in globals():
    original_dir = os.getcwd()

os.chdir(os.path.join(original_dir, "../../.."))
os.environ["DATA_DIR"] = os.path.join(os.getcwd(), "data")
os.environ["MODEL_DIR"] = os.path.join(os.getcwd(), "checkpoints")

with open("data/traces_dataset/nonograms/test.jsonl", "r") as f:
    json_dataset = [json.loads(line) for line in f.readlines()]
traces = [
    {"puzzle_idx": i, "method": m, **run}
    for i, p in enumerate(json_dataset)
    for m, d in p["traces"].items()
    for run in next(iter(d.values()))
]

df = pd.json_normalize(traces)
df.head()


### Getting a good example puzzle
- all models solved it
- took a few steps at least
- use the same puzzle for all of them

In [ ]:
solved = df[
    (df["stopping_reason"] == "SOLVED")
    & (df["num_steps"] > 4)
]

# puzzles solved (under the filter) by every method
n_methods = solved["method"].nunique()
shared_puzzles = (
    solved.groupby("puzzle_idx")["method"]
    .nunique()
    .pipe(lambda s: s[s == n_methods])
    .index
)

# pick one — e.g. the first shared puzzle
chosen = shared_puzzles[0]

examples = (
    solved[solved["puzzle_idx"] == chosen]
    .groupby("method")[["puzzle_idx", "steps", "num_steps"]]
    .first()
    .reset_index()
)
examples

In [ ]:
def evenly_sample(arr, m):
    m = min(m, len(arr))
    indices = np.linspace(0, len(arr) - 1, m).round().astype(int)
    arr = np.asarray(arr)
    return arr[indices], indices


def display_solver(clues, traces, width, height, title=None):
    total_elements = width * height
    traces, indices = evenly_sample(traces, total_elements)

    fig, axes = plt.subplots(height, width, figsize=(width * 3, height * 3))
    for i in range(width):
        for j in range(height):
            k = i * height + j
            ax = axes[k]
            display_grid(
                ax,
                traces[k],
                clues,
                title=f"step {indices[k]}"
            )
    fig.suptitle(title)
    fig.suptitle(title, fontsize=24, y=1.02)
    fig.tight_layout(rect=[0, 0, 1, 0.98])

In [ ]:
for row in examples.itertuples():
    display_solver(
        clues=json_dataset[row.puzzle_idx]["clues"],
        traces=row.steps,
        width=5,
        height=1,
        title=f"{row.method} (num_steps={row.num_steps})"
    )

In [ ]:

N_BUCKETS = 10
bucket_edges = np.linspace(0, 1, N_BUCKETS + 1)


def build_dist_matrix(step_grids):
    max_step = max(len(trace) for trace in step_grids) - 1
    matrix = np.zeros((max_step + 1, N_BUCKETS), dtype=np.int64)
    for trace in step_grids:
        for step_idx, grid in enumerate(trace):
            grid_arr = np.asarray(grid)
            counts, _ = np.histogram(grid_arr.ravel(), bins=bucket_edges)
            matrix[step_idx] += counts
    return matrix


def plot_distribution_heatmap(
    matrix,
    title,
    mode="full",
    max_steps=None,
):
    # --- Decide step range ---
    if mode == "full":
        matrix = matrix
        extent_x = matrix.shape[0]
        xlabel = "Step"

    elif mode == "max_steps":
        if max_steps is None:
            raise ValueError("mode='max_steps' requires max_steps")
        matrix = matrix[:max_steps]
        extent_x = matrix.shape[0]
        xlabel = f"Step (first {matrix.shape[0]})"

    else:
        raise ValueError(f"unknown mode: {mode}")

    # --- Normalize each row ---
    row_sums = matrix.sum(axis=1, keepdims=True)
    norm = np.divide(
        matrix, row_sums,
        out=np.zeros_like(matrix, dtype=float),
        where=row_sums != 0,
    )

    fig, ax = plt.subplots(figsize=(8, 5))
    im = ax.imshow(
        norm.T,
        aspect="auto",
        origin="lower",
        cmap="viridis",
        extent=[0, extent_x, 0, 1],
        vmin=0,
        vmax=1,
    )

    ax.set_xlabel(xlabel)
    ax.set_ylabel("Cell value bucket (0-1)")
    ax.set_title(title)
    plt.colorbar(im, label="proportion of cells")
    plt.tight_layout()
    plt.show()

In [ ]:

dist_matrices = {}
for method, group in df.groupby(["method"]):
    step_grids = group["steps"].tolist()  # list of traces (each trace = list of grids)
    dist_matrices[method] = build_dist_matrix(step_grids)


for method, matrix in dist_matrices.items():
    plot_distribution_heatmap(matrix, method, mode="max_steps", max_steps=50)
